In [55]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/datathon-2026-round-1/products.csv
/kaggle/input/competitions/datathon-2026-round-1/sample_submission.csv
/kaggle/input/competitions/datathon-2026-round-1/promotions.csv
/kaggle/input/competitions/datathon-2026-round-1/shipments.csv
/kaggle/input/competitions/datathon-2026-round-1/order_items.csv
/kaggle/input/competitions/datathon-2026-round-1/reviews.csv
/kaggle/input/competitions/datathon-2026-round-1/inventory.csv
/kaggle/input/competitions/datathon-2026-round-1/returns.csv
/kaggle/input/competitions/datathon-2026-round-1/sales.csv
/kaggle/input/competitions/datathon-2026-round-1/orders.csv
/kaggle/input/competitions/datathon-2026-round-1/geography.csv
/kaggle/input/competitions/datathon-2026-round-1/customers.csv
/kaggle/input/competitions/datathon-2026-round-1/baseline.ipynb
/kaggle/input/competitions/datathon-2026-round-1/payments.csv
/kaggle/input/competitions/datathon-2026-round-1/web_traffic.csv


In [56]:
import pandas as pd
import numpy as np

DATA_PATH = "/kaggle/input/competitions/datathon-2026-round-1/"

# ======================
# 1. LOAD DATA (fix dtype ngay từ đầu)
# ======================

products = pd.read_csv(DATA_PATH + "products.csv")
customers = pd.read_csv(DATA_PATH + "customers.csv", parse_dates=["signup_date"])
promotions = pd.read_csv(DATA_PATH + "promotions.csv", parse_dates=["start_date", "end_date"])
geography = pd.read_csv(DATA_PATH + "geography.csv")

orders = pd.read_csv(DATA_PATH + "orders.csv", parse_dates=["order_date"])
order_items = pd.read_csv(
    DATA_PATH + "order_items.csv",
    dtype={"promo_id": "string", "promo_id_2": "string"}
)
payments = pd.read_csv(DATA_PATH + "payments.csv")
shipments = pd.read_csv(DATA_PATH + "shipments.csv", parse_dates=["ship_date", "delivery_date"])
returns = pd.read_csv(DATA_PATH + "returns.csv", parse_dates=["return_date"])
reviews = pd.read_csv(DATA_PATH + "reviews.csv", parse_dates=["review_date"])

sales = pd.read_csv(DATA_PATH + "sales.csv", parse_dates=["Date"])
inventory = pd.read_csv(DATA_PATH + "inventory.csv", parse_dates=["snapshot_date"])
web_traffic = pd.read_csv(DATA_PATH + "web_traffic.csv", parse_dates=["date"])

# ======================
# 2. CLEAN COLUMN NAME (optional nhưng nên làm)
# ======================

def clean_columns(df):
    df.columns = df.columns.str.strip().str.lower()
    return df

products = clean_columns(products)
customers = clean_columns(customers)
promotions = clean_columns(promotions)
geography = clean_columns(geography)
orders = clean_columns(orders)
order_items = clean_columns(order_items)
payments = clean_columns(payments)
shipments = clean_columns(shipments)
returns = clean_columns(returns)
reviews = clean_columns(reviews)
sales = clean_columns(sales)
inventory = clean_columns(inventory)
web_traffic = clean_columns(web_traffic)

# ======================
# 3. FIX KEY CONFLICT (zip rất dễ lỗi)
# ======================

orders = orders.rename(columns={"zip": "order_zip"})
customers = customers.rename(columns={"zip": "customer_zip"})

# ======================
# 4. HANDLE MISSING VALUES
# ======================

# customers
customers["gender"] = customers["gender"].fillna("unknown")
customers["age_group"] = customers["age_group"].fillna("unknown")
customers["acquisition_channel"] = customers["acquisition_channel"].fillna("unknown")

# order_items
order_items["promo_id"] = order_items["promo_id"].fillna("none")
order_items["promo_id_2"] = order_items["promo_id_2"].fillna("none")

# promotions
promotions["applicable_category"] = promotions["applicable_category"].fillna("all")

# ======================
# 5. BASIC FEATURE ENGINEERING
# ======================

# products
products["margin"] = (products["price"] - products["cogs"]) / products["price"]

# orders (time features)
orders["year"] = orders["order_date"].dt.year
orders["month"] = orders["order_date"].dt.month
orders["dayofweek"] = orders["order_date"].dt.dayofweek

# ======================
# 6. BUILD CORE FACT TABLE
# ======================

df = (
    order_items
    .merge(orders, on="order_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(geography, left_on="order_zip", right_on="zip", how="left")
)

# ======================
# 7. ADD BUSINESS METRICS
# ======================

df["revenue"] = df["quantity"] * df["unit_price"]
df["cost"] = df["quantity"] * df["cogs"]
df["profit"] = df["revenue"] - df["cost"]
df["margin"] = df["profit"] / df["revenue"]

df["has_promo"] = (df["promo_id"] != "none").astype(int)

# ======================
# 8. DATA SANITY CHECK
# ======================

print("Final shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nSample:")
df.head()

Final shape: (714669, 39)

Missing values:
order_id           0
product_id         0
quantity           0
unit_price         0
discount_amount    0
promo_id           0
promo_id_2         0
order_date         0
customer_id        0
order_zip          0
dtype: int64

Sample:


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,order_date,customer_id,order_zip,...,age_group,acquisition_channel,zip,city_y,region,district,revenue,cost,profit,has_promo
0,1,2400,7,1138.22,0.0,none,none,2012-07-04,58578,1109,...,35-44,social_media,1109,Hanoi,East,District #02,7967.54,7376.586059,590.953941,0
1,2,609,7,10166.25,0.0,none,none,2012-07-04,58621,1330,...,18-24,social_media,1330,Phu Ly,East,District #02,71163.75,62913.929616,8249.820384,0
2,3,396,3,11220.33,0.0,none,none,2012-07-04,58811,1473,...,35-44,direct,1473,Lao Cai,East,District #02,33660.99,30273.036767,3387.953233,0
3,4,635,5,10639.25,0.0,none,none,2012-07-04,59453,2360,...,45-54,direct,2360,Son Tay,East,District #02,53196.25,46027.152390,7169.097610,0
4,6,1935,1,1597.84,0.0,none,none,2012-07-06,57821,2886,...,18-24,social_media,2886,Uong Bi,East,District #02,1597.84,1048.696357,549.143643,0


In [57]:
sales.head()

,date,revenue,cogs
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79


In [58]:
# ==== Rename trước để tránh xung đột cột ====
orders = orders.rename(columns={"zip": "order_zip"})
customers = customers.rename(columns={"zip": "customer_zip"})

# ==== Merge pipeline ====
df = (
    order_items
    .merge(orders, on="order_id", how="left")              # thêm thông tin đơn hàng
    .merge(products, on="product_id", how="left")          # thêm thông tin sản phẩm
    .merge(customers, on="customer_id", how="left")        # thêm thông tin khách hàng
    .merge(geography, left_on="order_zip", right_on="zip", how="left")  # địa lý theo nơi giao hàng
)

# ==== Check ====
print(df.shape)
df.head()

(714669, 35)


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,order_date,customer_id,order_zip,...,customer_zip,city_x,signup_date,gender,age_group,acquisition_channel,zip,city_y,region,district
0,1,2400,7,1138.22,0.0,none,none,2012-07-04,58578,1109,...,1109,Hanoi,2020-06-06,Female,35-44,social_media,1109,Hanoi,East,District #02
1,2,609,7,10166.25,0.0,none,none,2012-07-04,58621,1330,...,1330,Phu Ly,2021-11-03,Female,18-24,social_media,1330,Phu Ly,East,District #02
2,3,396,3,11220.33,0.0,none,none,2012-07-04,58811,1473,...,1473,Lao Cai,2020-09-18,Female,35-44,direct,1473,Lao Cai,East,District #02
3,4,635,5,10639.25,0.0,none,none,2012-07-04,59453,2360,...,2360,Son Tay,2016-05-29,Male,45-54,direct,2360,Son Tay,East,District #02
4,6,1935,1,1597.84,0.0,none,none,2012-07-06,57821,2886,...,2886,Uong Bi,2017-07-11,Male,18-24,social_media,2886,Uong Bi,East,District #02


# Questions

In [59]:
import pandas as pd
### Q1
# đảm bảo order_date là datetime
orders["order_date"] = pd.to_datetime(orders["order_date"])

# sort theo customer + thời gian
orders_sorted = orders.sort_values(["customer_id", "order_date"])

# tính ngày trước đó
orders_sorted["prev_order_date"] = (
    orders_sorted.groupby("customer_id")["order_date"].shift(1)
)

# tính gap (số ngày)
orders_sorted["gap_days"] = (
    orders_sorted["order_date"] - orders_sorted["prev_order_date"]
).dt.days

# ❗ chỉ giữ customer có >1 order
multi_order_customers = (
    orders.groupby("customer_id")
    .size()
    .loc[lambda x: x > 1]
    .index
)

df_gap = orders_sorted[
    orders_sorted["customer_id"].isin(multi_order_customers)
]

# drop NaN (dòng đầu mỗi customer)
df_gap = df_gap.dropna(subset=["gap_days"])

# tính median
median_gap = df_gap["gap_days"].median()

print("Median inter-order gap:", median_gap)

Median inter-order gap: 144.0


In [60]:
# Q2: tính margin cho từng product
products["margin"] = (products["price"] - products["cogs"]) / products["price"]

# group theo segment
segment_margin = (
    products
    .groupby("segment")["margin"]
    .mean()
    .sort_values(ascending=False)
)

print(segment_margin)

# lấy segment cao nhất
best_segment = segment_margin.idxmax()
print("\nBest segment:", best_segment)

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: margin, dtype: float64

Best segment: Standard


In [61]:
# join returns với products
df_return = returns.merge(products, on="product_id", how="left")

# lọc category = Streetwear
df_streetwear = df_return[df_return["category"] == "Streetwear"]

# đếm lý do trả hàng
reason_counts = (
    df_streetwear["return_reason"]
    .value_counts()
)

print(reason_counts)

# lấy lý do nhiều nhất
top_reason = reason_counts.idxmax()
print("\nMost common return reason:", top_reason)

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Most common return reason: wrong_size


In [62]:
# tính bounce_rate trung bình theo từng nguồn
bounce_avg = (
    web_traffic
    .groupby("traffic_source")["bounce_rate"]
    .mean()
    .sort_values()
)

print(bounce_avg)

# lấy source thấp nhất
best_source = bounce_avg.idxmin()
print("\nLowest bounce rate source:", best_source)

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Lowest bounce rate source: email_campaign


In [63]:
promo_ratio = (order_items["promo_id"] != "none").mean() * 100
promo_ratio

np.float64(38.663493169565214)

In [64]:
# lọc age_group != null
customers_valid = customers[customers["age_group"].notnull()]

# merge orders
df_cust_orders = orders.merge(customers_valid, on="customer_id", how="inner")

# số đơn theo từng nhóm tuổi
orders_per_group = df_cust_orders.groupby("age_group")["order_id"].count()

# số khách theo từng nhóm tuổi
customers_per_group = customers_valid.groupby("age_group")["customer_id"].nunique()

# trung bình
avg_orders = (orders_per_group / customers_per_group).sort_values(ascending=False)

print(avg_orders)

best_age_group = avg_orders.idxmax()
print("\nBest age group:", best_age_group)

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
dtype: float64

Best age group: 55+


In [65]:
# dùng df đã build từ preprocessing (order_items + orders + geography)

region_revenue = (
    df.groupby("region")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

print(region_revenue)

best_region = region_revenue.idxmax()
print("\nBest region:", best_region)

KeyError: 'Column not found: revenue'

In [ ]:
cancel_orders = orders[orders["order_status"] == "cancelled"]

payment_counts = cancel_orders["payment_method"].value_counts()

print(payment_counts)

top_payment = payment_counts.idxmax()
print("\nMost used payment method (cancelled):", top_payment)

In [ ]:
# order_items + products → lấy size
oi = order_items.merge(products, on="product_id", how="left")

# returns + products → lấy size
rt = returns.merge(products, on="product_id", how="left")

# chỉ giữ S, M, L, XL
valid_sizes = ["S", "M", "L", "XL"]
oi = oi[oi["size"].isin(valid_sizes)]
rt = rt[rt["size"].isin(valid_sizes)]

# số dòng order_items theo size
oi_count = oi["size"].value_counts()

# số dòng returns theo size
rt_count = rt["size"].value_counts()

# align index
return_rate = (rt_count / oi_count).sort_values(ascending=False)

print(return_rate)

best_size = return_rate.idxmax()
print("\nHighest return rate size:", best_size)

In [ ]:
avg_payment = (
    payments
    .groupby("installments")["payment_value"]
    .mean()
    .sort_values(ascending=False)
)

print(avg_payment)

best_plan = avg_payment.idxmax()
print("\nBest installment plan:", best_plan)

# EDA

Gồm 6 cụm phân tích
### Cụm A — Revenue & seasonality

Mục tiêu: cho người chấm thấy bức tranh tổng thể.

Doanh thu theo thời gian
Doanh thu theo tháng / quý / năm
Growth rate, seasonality, peak months
So sánh Revenue với COGS để nhìn gross trend

Insight nên nhắm tới: có mùa cao điểm không, tăng trưởng có đều không, giai đoạn nào bất thường.

Chart hợp lý: line chart, rolling mean, heatmap month-year.

### Cụm B — Customer behavior

Mục tiêu: hiểu ai mua hàng và họ mua như thế nào.

Inter-order gap
Repeat rate vs new customers
Orders per customer theo age_group, gender, acquisition_channel
Cohort đơn giản theo signup_date hoặc order_date

Insight nên nhắm tới: nhóm khách nào mua nhiều, nhóm nào quay lại tốt, kênh nào tạo khách chất lượng.

Chart hợp lý: bar chart, boxplot, cohort heatmap.

### Cụm C — Product, margin, and pricing

Mục tiêu: nhìn doanh thu dưới góc sản phẩm, không chỉ tổng tiền.

Margin theo segment/category
Revenue, quantity, profit theo category/segment/size/color
High revenue nhưng low margin vs low revenue nhưng high margin

Insight nên nhắm tới: segment nào đang “bán nhiều nhưng lời ít”, segment nào là mỏ vàng thật sự.

Chart hợp lý: grouped bar, scatter revenue vs margin, Pareto chart.

### Cụm D — Promotions and payment

Mục tiêu: hiểu khuyến mãi có tạo giá trị thật hay chỉ đốt margin.

Tỷ lệ dòng có promo
Revenue / quantity / margin theo promo vs non-promo
promo_type, stackable_flag, min_order_value
payment_method, installments và giá trị đơn hàng

Insight nên nhắm tới: khuyến mãi nào có lợi, loại thanh toán nào liên quan đến đơn lớn hơn, trả góp có làm tăng giá trị đơn không.

Chart hợp lý: boxplot, stacked bar, violin plot.

### E — Returns and reviews

Mục tiêu: chỉ ra nơi doanh nghiệp đang mất tiền âm thầm.

Return rate theo size, category, segment
Return reason theo category
Rating theo product/category/segment
Correlation giữa rating thấp và return cao

Insight nên nhắm tới: size nào rủi ro nhất, category nào có vấn đề fit/chất lượng, return có tập trung vào một lý do cụ thể không.

Chart hợp lý: bar chart, heatmap, sankey nhẹ nếu cần kể chuyện tốt.

### Cụm F — Traffic, inventory, and operational impact

Đây là cụm nên làm để khác biệt với các đội khác.

Traffic source vs bounce rate
Sessions / unique visitors vs revenue
Inventory stockout_days, fill_rate, reorder_flag vs sales
Stockout có đi kèm giảm doanh thu không
Web traffic tăng nhưng revenue không tăng: vấn đề nằm ở conversion hay supply?

Insight nên nhắm tới: nguồn traffic nào chất lượng, stockout có làm mất doanh thu không, vận hành có đang bóp hiệu quả marketing không.

Chart hợp lý: line chart đồng trục, scatter, lag correlation, small multiple.

## Basic EDA

In [ ]:
df.describe()

In [ ]:
df.isnull().mean().sort_values()

In [ ]:
df.nunique()

In [ ]:
df.groupby("category")["revenue"].sum()

## Cụm A

In [ ]:
sales = sales.sort_values("date")
sales = sales.set_index("date")
import matplotlib.pyplot as plt

plt.figure(figsize=(14,5))
plt.plot(sales["revenue"], label="Revenue")
plt.title("Revenue over Time")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.show()

In [ ]:
sales["rolling_30"] = sales["revenue"].rolling(30).mean()
plt.figure(figsize=(14,5))
plt.plot(sales["revenue"], alpha=0.3, label="Doanh thu theo ngày")
plt.plot(sales["rolling_30"], color="red", label="Trung bình trượt 30 ngày")

plt.title("Xu hướng doanh thu theo thời gian")
plt.xlabel("Thời gian")
plt.ylabel("Doanh thu")
plt.legend()
plt.show()

In [ ]:
# seasonality theo tháng
# Biểu đồ thể hiện doanh thu trung bình theo từng tháng trong năm. Mục tiêu là xác định yếu tố mùa vụ (seasonality) trong hành vi mua sắm

sales["month"] = sales.index.month
monthly_avg = sales.groupby("month")["revenue"].mean()

monthly_avg.plot(kind="bar")

plt.title("Doanh thu trung bình theo tháng")
plt.xlabel("Tháng")
plt.ylabel("Doanh thu trung bình")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# year over yera
#Doanh thu tăng mạnh từ ~0.7B (2012) lên ~2.1B (2016).
#Sau đó giảm dần xuống khoảng ~1.0–1.2B trong giai đoạn 2019–2021.
#Năm 2022 có dấu hiệu phục hồi nhẹ nhưng vẫn chưa quay lại đỉnh cũ.
sales["year"] = sales.index.year
yearly = sales.groupby("year")["revenue"].sum()

yearly.plot(kind="bar")

plt.title("Tổng doanh thu theo năm")
plt.xlabel("Năm")
plt.ylabel("Doanh thu")
plt.xticks(rotation=0)
plt.show()

## Cụm B

In [ ]:
# Tần suất mua
orders_sorted = orders.sort_values(["customer_id", "order_date"])

orders_sorted["prev_date"] = orders_sorted.groupby("customer_id")["order_date"].shift(1)
orders_sorted["gap_days"] = (orders_sorted["order_date"] - orders_sorted["prev_date"]).dt.days

gap = orders_sorted["gap_days"].dropna()

print("Median gap:", gap.median())
import seaborn as sns
sns.histplot(gap, bins=50)

plt.title("Phân phối khoảng thời gian giữa hai lần mua")
plt.xlabel("Số ngày giữa hai lần mua")
plt.ylabel("Số lượng khách hàng")
plt.show()

In [ ]:
customer_revenue = df.groupby("customer_id")["revenue"].sum().sort_values(ascending=False)

cum_pct = customer_revenue.cumsum() / customer_revenue.sum()

plt.figure(figsize=(10,5))
plt.plot(cum_pct.values)

plt.axhline(0.8, linestyle="--")
plt.title("Tỷ lệ đóng góp doanh thu tích lũy theo khách hàng")
plt.xlabel("Khách hàng (xếp theo doanh thu giảm dần)")
plt.ylabel("Tỷ lệ doanh thu tích lũy")
plt.show()

In [ ]:
customer_orders = orders.groupby("customer_id")["order_id"].nunique()
customer_orders.describe()

In [ ]:
customer_orders = orders.groupby("customer_id")["order_id"].nunique()

customer_orders.value_counts().head(10).plot(kind="bar")

plt.title("Phân phối số đơn hàng trên mỗi khách")
plt.xlabel("Số đơn hàng")
plt.ylabel("Số lượng khách")
plt.show()

In [ ]:
df_orders = orders.merge(customers, on="customer_id")

orders_per_customer = (
    df_orders.groupby(["age_group", "customer_id"])["order_id"]
    .nunique()
    .groupby("age_group")
    .mean()
    .sort_values(ascending=False)
)

orders_per_customer.plot(kind="bar")

plt.title("Số đơn hàng trung bình theo nhóm tuổi")
plt.xlabel("Nhóm tuổi")
plt.ylabel("Số đơn hàng trung bình")
plt.show()

In [ ]:
## Cái trên cho thấy trung bình giữa các nhóm tuổi khá tương đương nhau -> cần viết tổng quan rồi tìm ra insight của plot sau
import seaborn as sns

sns.boxplot(
    data=df_orders.groupby(["age_group", "customer_id"])["order_id"]
    .nunique()
    .reset_index(),
    x="age_group",
    y="order_id"
)

In [ ]:
df_orders = orders.merge(customers, on="customer_id")

channel_perf = (
    df_orders.groupby(["acquisition_channel", "customer_id"])["order_id"]
    .nunique()
    .groupby("acquisition_channel")
    .mean()
    .sort_values(ascending=False)
)
channel_perf.plot(kind="bar")

plt.title("Hiệu quả khách hàng theo kênh tiếp cận")
plt.xlabel("Kênh tiếp cận")
plt.ylabel("Số đơn hàng trung bình trên mỗi khách")
plt.xticks(rotation=30)
plt.show()

In [ ]:
import seaborn as sns

cust_orders = (
    df_orders.groupby(["acquisition_channel", "customer_id"])["order_id"]
    .nunique()
    .reset_index()
)

sns.boxplot(
    data=cust_orders,
    x="acquisition_channel",
    y="order_id"
)

plt.title("Phân phối số đơn hàng theo kênh tiếp cận")
plt.xlabel("Kênh tiếp cận")
plt.ylabel("Số đơn hàng mỗi khách")
plt.xticks(rotation=30)
plt.show()

In [ ]:
orders["order_month"] = orders["order_date"].dt.to_period("M")
customers["signup_month"] = customers["signup_date"].dt.to_period("M")

df_cohort = orders.merge(customers, on="customer_id")

cohort = df_cohort.groupby(["signup_month", "order_month"])["customer_id"].nunique().unstack()

import seaborn as sns
sns.heatmap(cohort, cmap="Blues")

plt.title("Cohort khách hàng theo thời gian")
plt.xlabel("Tháng mua hàng")
plt.ylabel("Tháng đăng ký")
plt.show()

## Cụm C

In [ ]:
# product-level aggregate
# Cách viết phân tích dễ hiểu
## Mô tả: Biểu đồ so sánh doanh thu và lợi nhuận theo từng phân khúc sản phẩm. Mục đích là xem phân khúc nào thật sự tạo ra tiền, chứ không chỉ tạo ra doanh số.
## Key findings: Nếu có phân khúc doanh thu cao nhưng lợi nhuận không tương xứng, đó là dấu hiệu “bán nhiều nhưng lời mỏng”. Ngược lại, phân khúc có doanh thu vừa phải nhưng lợi nhuận tốt là “mỏ vàng thật sự”.
## Business implication: Không nên chỉ đẩy phân khúc có doanh thu cao; cần ưu tiên phân khúc có biên lợi nhuận tốt và doanh thu ổn định.

prod = (
    df.groupby(["product_id", "product_name", "category", "segment", "size", "color"], as_index=False)
      .agg(
          revenue=("revenue", "sum"),
          quantity=("quantity", "sum"),
          profit=("profit", "sum")
      )
)

prod["margin"] = prod["profit"] / prod["revenue"]

seg = (
    prod.groupby("segment", as_index=False)
        .agg(
            revenue=("revenue", "sum"),
            profit=("profit", "sum"),
            quantity=("quantity", "sum"),
            margin=("margin", "mean")
        )
        .sort_values("revenue", ascending=False)
)

import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(seg["segment"]))
w = 0.35

plt.figure(figsize=(12,5))
plt.bar(x - w/2, seg["revenue"], width=w, label="Doanh thu")
plt.bar(x + w/2, seg["profit"], width=w, label="Lợi nhuận")
plt.title("Doanh thu và lợi nhuận theo phân khúc")
plt.xlabel("Phân khúc")
plt.ylabel("Giá trị")
plt.xticks(x, seg["segment"], rotation=30)
plt.legend()
plt.tight_layout()
plt.show()

### Scatter — Revenue vs Margin theo product

Mục tiêu: tìm sản phẩm nào nằm ở góc “nguy hiểm”: doanh thu cao nhưng margin thấp.

In [ ]:
# Mục tiêu: tìm sản phẩm nào nằm ở góc “nguy hiểm”: doanh thu cao nhưng margin thấp.

plt.figure(figsize=(10,6))
plt.scatter(prod["revenue"], prod["margin"], alpha=0.6)
plt.title("Doanh thu và biên lợi nhuận theo sản phẩm")
plt.xlabel("Doanh thu")
plt.ylabel("Biên lợi nhuận")
plt.tight_layout()
plt.show()

### Tương tự plot trên nhưng chỉ annotate top sản phẩm theo revenue:

Mô tả: Biểu đồ cho thấy mối quan hệ giữa doanh thu và biên lợi nhuận của từng sản phẩm.

Key findings: Có thể xuất hiện bốn vùng rất rõ:

- doanh thu cao, margin cao: sản phẩm “ngôi sao”
- doanh thu cao, margin thấp: bán tốt nhưng đang ăn mòn lợi nhuận
- doanh thu thấp, margin cao: sản phẩm ngách nhưng đáng giữ
- doanh thu thấp, margin thấp: nên xem lại

Business implication: Đây là chart giúp ưu tiên hành động: giữ gìn sản phẩm tốt, điều chỉnh giá/khuyến mãi cho sản phẩm doanh thu cao nhưng margin thấp, và loại bỏ hoặc cải tiến sản phẩm kém hiệu quả.

In [ ]:
# Tương tự plot trên nhưng chỉ annotate top sản phẩm theo revenue:

top_products = prod.nlargest(10, "revenue")

plt.figure(figsize=(10,6))
plt.scatter(prod["revenue"], prod["margin"], alpha=0.5)
plt.scatter(top_products["revenue"], top_products["margin"], color="red")

for _, r in top_products.iterrows():
    plt.text(r["revenue"], r["margin"], r["product_name"], fontsize=8)

plt.title("Doanh thu và biên lợi nhuận theo sản phẩm")
plt.xlabel("Doanh thu")
plt.ylabel("Biên lợi nhuận")
plt.tight_layout()
plt.show()

### Chart 3: Pareto chart — doanh thu tích lũy theo sản phẩm

Mục tiêu: xem doanh thu có bị phụ thuộc vào một số ít SKU hay không.

Cách viết phân tích

- Mô tả: Biểu đồ cho biết doanh thu có tập trung vào một nhóm nhỏ sản phẩm hay không.
- Key findings: Nếu đường tích lũy tăng rất nhanh, nghĩa là một số ít sản phẩm đang đóng góp phần lớn doanh thu.
- Business implication: Doanh nghiệp cần bảo vệ nhóm SKU chủ lực này: ưu tiên tồn kho, theo dõi giá, tránh stockout, và không để khuyến mãi làm hỏng biên lợi nhuận.

In [ ]:
pareto = prod.sort_values("revenue", ascending=False).reset_index(drop=True)
pareto["cum_revenue_pct"] = pareto["revenue"].cumsum() / pareto["revenue"].sum()

fig, ax1 = plt.subplots(figsize=(12,5))
ax1.bar(range(len(pareto)), pareto["revenue"], alpha=0.7)
ax1.set_title("Pareto doanh thu theo sản phẩm")
ax1.set_xlabel("Sản phẩm (xếp theo doanh thu giảm dần)")
ax1.set_ylabel("Doanh thu")

ax2 = ax1.twinx()
ax2.plot(range(len(pareto)), pareto["cum_revenue_pct"], color="red")
ax2.axhline(0.8, linestyle="--")
ax2.set_ylabel("Tỷ lệ doanh thu tích lũy")

plt.tight_layout()
plt.show()

margin theo category

In [ ]:
cat = (
    prod.groupby("category", as_index=False)
        .agg(
            revenue=("revenue", "sum"),
            profit=("profit", "sum")
        )
)
cat["margin"] = cat["profit"] / cat["revenue"]

plt.figure(figsize=(10,4))
plt.bar(cat["category"], cat["margin"])
plt.title("Biên lợi nhuận theo danh mục")
plt.xlabel("Danh mục")
plt.ylabel("Biên lợi nhuận")
plt.tight_layout()
plt.show()

**Story telling cho cụm C**

có thể viết theo logic này:

Phân tích sản phẩm cho thấy không phải phân khúc nào tạo ra doanh thu cao cũng mang lại lợi nhuận tốt. Một số sản phẩm hoặc phân khúc có doanh thu lớn nhưng biên lợi nhuận thấp, cho thấy doanh nghiệp đang tăng trưởng bằng cách hy sinh lợi nhuận. Ngược lại, một số phân khúc có doanh thu vừa phải nhưng margin tốt hơn, phù hợp để ưu tiên mở rộng.

Pareto chart cũng cho thấy doanh thu có thể phụ thuộc mạnh vào một nhóm nhỏ SKU chủ lực. Điều này khiến doanh nghiệp dễ tổn thương nếu nhóm sản phẩm này gặp lỗi tồn kho, giảm sức mua, hoặc bị cạnh tranh.

Vì vậy, chiến lược hợp lý không chỉ là đẩy doanh số, mà là tối ưu đồng thời doanh thu và biên lợi nhuận: giữ SKU “ngôi sao”, chỉnh giá hoặc khuyến mãi cho SKU lời mỏng, và giảm nguồn lực cho SKU kém hiệu quả.

## Cụm D

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

promo_data = df.copy()

# vì trước đó đã fill "none", đổi lại để merge cho sạch
promo_data["promo_id"] = promo_data["promo_id"].replace("none", pd.NA)
promo_data["promo_id_2"] = promo_data["promo_id_2"].replace("none", pd.NA)

# nối thêm thông tin khuyến mãi
promo_data = promo_data.merge(
    promotions[["promo_id", "promo_type", "discount_value", "stackable_flag", "min_order_value"]],
    on="promo_id",
    how="left"
)

# nối thêm thanh toán
promo_data = promo_data.merge(
    payments[["order_id", "payment_value", "installments"]],
    on="order_id",
    how="left"
)

# feature cơ bản
promo_data["promo_group"] = np.where(promo_data["promo_id"].notna(), "Có khuyến mãi", "Không khuyến mãi")
promo_data["actual_margin"] = promo_data["profit"] / promo_data["revenue"]
promo_data["gross_value"] = promo_data["quantity"] * promo_data["price"]
promo_data["discount_rate"] = promo_data["discount_amount"] / promo_data["gross_value"]
promo_data["stackable_label"] = promo_data["stackable_flag"].map({1: "Stackable", 0: "Không stackable"})
promo_data["min_order_label"] = np.where(promo_data["min_order_value"].notna(), "Có ngưỡng tối thiểu", "Không có ngưỡng")

In [ ]:
promo_rate = promo_data["promo_id"].notna().mean() * 100
print(f"Tỷ lệ dòng có promo: {promo_rate:.2f}%")
#Gợi ý viết phân tích: chỉ số này cho biết khuyến mãi có phải là công cụ phổ biến hay chỉ dùng lẻ tẻ. Nếu tỷ lệ này cao, nghĩa là doanh nghiệp đang dựa khá nhiều vào giảm giá để bán hàng. Nếu thấp, promo chỉ là đòn bẩy phụ, không phải động cơ chính.

### Boxplot cho nhóm khuyến mãi
Cách viết phân tích

Biểu đồ này trả lời câu hỏi rất thực tế: khuyến mãi có làm bán được nhiều hơn không, và cái giá phải trả là gì. Nếu nhóm có khuyến mãi có doanh thu và số lượng cao hơn nhưng biên lợi nhuận thấp hơn, thì khuyến mãi đang giúp tăng volume nhưng làm mỏng lợi nhuận. Đây là kiểu “bán khỏe nhưng lời mỏng”.

Business implication

Doanh nghiệp không nên nhìn promo chỉ qua doanh thu. Cần theo dõi thêm margin để biết giảm giá nào thật sự đáng làm.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.boxplot(data=promo_data, x="promo_group", y="revenue", ax=axes[0])
axes[0].set_title("Doanh thu theo nhóm khuyến mãi")
axes[0].set_xlabel("")
axes[0].set_ylabel("Doanh thu")

sns.boxplot(data=promo_data, x="promo_group", y="quantity", ax=axes[1])
axes[1].set_title("Số lượng theo nhóm khuyến mãi")
axes[1].set_xlabel("")
axes[1].set_ylabel("Số lượng")

sns.boxplot(data=promo_data, x="promo_group", y="actual_margin", ax=axes[2])
axes[2].set_title("Biên lợi nhuận theo nhóm khuyến mãi")
axes[2].set_xlabel("")
axes[2].set_ylabel("Biên lợi nhuận")

plt.tight_layout()
plt.show()

### Violin plot 
Cách viết phân tích

Biểu đồ này giúp so sánh các loại khuyến mãi khác nhau. Nếu một loại promo nào đó có phân phối margin thấp hơn rõ rệt, nghĩa là nó đang “đốt” lợi nhuận mạnh hơn các loại khác. Nếu promo stackable cũng kéo margin xuống sâu hơn, doanh nghiệp nên kiểm soát việc cộng dồn ưu đãi.

Business implication

Có thể giữ loại khuyến mãi mang lại hiệu quả tốt, nhưng nên hạn chế loại làm margin tụt mạnh. Nói thẳng: promo nào chỉ giỏi phá giá thì nên cho nó nghỉ.

In [ ]:
promo_only = promo_data[promo_data["promo_group"] == "Có khuyến mãi"].copy()

plt.figure(figsize=(12, 5))
sns.violinplot(
    data=promo_only,
    x="promo_type",
    y="actual_margin",
    hue="stackable_label",
    split=True,
    inner="quartile"
)
plt.title("Biên lợi nhuận theo loại khuyến mãi và khả năng stack")
plt.xlabel("Loại khuyến mãi")
plt.ylabel("Biên lợi nhuận")
plt.legend(title="")
plt.tight_layout()
plt.show()

### Stack 
Cách viết phân tích

Biểu đồ này cho biết doanh nghiệp đang dùng promo như thế nào về mặt thiết kế chiến dịch. Nếu khuyến mãi stackable xuất hiện nhiều, doanh nghiệp đang ưu tiên kích cầu mạnh. Nhưng nếu nhóm này cũng gắn với margin thấp, thì cần siết lại. min_order_value cũng là một “van điều tiết” quan trọng: đặt ngưỡng tối thiểu giúp đẩy giá trị đơn hàng lên mà không phải giảm giá vô tội vạ.

Business implication

Stacking và ngưỡng tối thiểu nên được dùng như công cụ kiểm soát, không phải để giảm giá bừa.

In [ ]:
stack_summary = (
    promo_only
    .groupby(["promo_type", "stackable_label"])
    .size()
    .reset_index(name="count")
)

stack_pivot = stack_summary.pivot(index="promo_type", columns="stackable_label", values="count").fillna(0)

stack_pivot.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.title("Số lượng khuyến mãi theo loại và khả năng stack")
plt.xlabel("Loại khuyến mãi")
plt.ylabel("Số lượng")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Boxplot: payment_value theo installments
Cách viết phân tích

Biểu đồ này kiểm tra xem trả góp có liên quan đến đơn hàng lớn hơn không. Nếu số kỳ trả góp cao đi kèm payment_value cao hơn, thì trả góp có thể đang giúp khách chấp nhận đơn hàng giá trị lớn. Nếu không có khác biệt rõ, trả góp chỉ là phương thức thanh toán chứ không phải đòn bẩy bán hàng.

Business implication

Nếu trả góp thực sự gắn với đơn lớn hơn, doanh nghiệp có thể dùng nó cho sản phẩm giá cao. Nếu không, không nên coi trả góp là công cụ tăng doanh thu chính.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=promo_data, x="installments", y="payment_value")
plt.title("Giá trị thanh toán theo số kỳ trả góp")
plt.xlabel("Số kỳ trả góp")
plt.ylabel("Giá trị thanh toán")
plt.tight_layout()
plt.show()

## Cụm E

In [ ]:
rr = df.copy()

# gắn cờ return (nếu bạn đã merge returns trước đó thì giữ lại cột is_return)
if "is_return" not in rr.columns:
    returns_flag = returns.assign(is_return=1)[["order_id","product_id","is_return"]]
    rr = rr.merge(returns_flag, on=["order_id","product_id"], how="left")
    rr["is_return"] = rr["is_return"].fillna(0)

# gắn rating
rr = rr.merge(
    reviews[["order_id","product_id","rating"]],
    on=["order_id","product_id"],
    how="left"
)

# tránh NaN chia cho 0
rr["is_return"] = rr["is_return"].astype(int)

returns_flag = returns.copy()
returns_flag["is_return"] = 1

df = df.merge(
    returns_flag[["order_id", "product_id", "is_return"]],
    on=["order_id", "product_id"],
    how="left"
)

df["is_return"] = df["is_return"].fillna(0)

In [ ]:
ret = (
    df.groupby(["category","segment"])
    .agg(
        orders=("order_id","count"),
        returns=("is_return","sum")
    )
    .reset_index()
)

ret["return_rate"] = ret["returns"] / ret["orders"]

heat = ret.pivot(index="category", columns="segment", values="return_rate")

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="Reds")

plt.title("Tỷ lệ trả hàng theo Category và Segment")
plt.xlabel("Segment")
plt.ylabel("Category")
plt.show()

In [ ]:
reason = returns["return_reason"].value_counts()

reason.plot(kind="bar")

plt.title("Lý do trả hàng")
plt.xlabel("Lý do")
plt.ylabel("Số lượng")
plt.xticks(rotation=30)
plt.show()

## Cụm F

In [ ]:
print(sales.index.name)
print(sales.columns)

In [ ]:
tr = web_traffic.sort_values("date").set_index("date")
sl = sales.sort_index()#.set_index("date")

tv = sl.join(tr[["sessions"]], how="inner")

# scale sessions để vẽ chung trục (chuẩn hóa min-max)
tv["sessions_scaled"] = (tv["sessions"] - tv["sessions"].min()) / (tv["sessions"].max() - tv["sessions"].min())
tv["revenue_scaled"]  = (tv["revenue"]  - tv["revenue"].min())  / (tv["revenue"].max()  - tv["revenue"].min())

import matplotlib.pyplot as plt
plt.figure(figsize=(14,5))
plt.plot(tv.index, tv["revenue_scaled"], label="Doanh thu (chuẩn hóa)")
plt.plot(tv.index, tv["sessions_scaled"], label="Web Traffic (chuẩn hóa)")
plt.title("Web Traffic vs Doanh thu theo thời gian")
plt.xlabel("Thời gian")
plt.ylabel("Giá trị chuẩn hóa")
plt.legend()
plt.tight_layout()
plt.show()

# Model

In [80]:
DIR = "/kaggle/input/competitions/datathon-2026-round-1/"

In [81]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from prophet import Prophet
from sklearn.metrics import mean_absolute_error
import optuna
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [82]:
SALE_SEASONS = [
    {'month': 1,  'start_day': 30, 'duration': 30, 'profit_rank': 1},
    {'month': 3,  'start_day': 18, 'duration': 30, 'profit_rank': 2},
    {'month': 6,  'start_day': 23, 'duration': 29, 'profit_rank': 3},
    {'month': 7,  'start_day': 30, 'duration': 34, 'profit_rank': 5},
    {'month': 8,  'start_day': 30, 'duration': 32, 'profit_rank': 4},
    {'month': 11, 'start_day': 18, 'duration': 45, 'profit_rank': 6},
]

In [83]:
def add_seasonality_features(df):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    d = df["Date"]
    df['year']      = d.dt.year
    df['month']     = d.dt.month
    df['day']       = d.dt.day
    df['dayofweek'] = d.dt.dayofweek
    df['doy']       = d.dt.dayofyear
    df['is_peak_season'] = d.dt.month.isin([4, 5, 6]).astype(int)
    def days_to_nearest_peak(date):
        y = date.year
        candidates = [
            pd.Timestamp(y,4,15),pd.Timestamp(y,5,15),pd.Timestamp(y,6,15),
            pd.Timestamp(y-1,4,15),pd.Timestamp(y-1,5,15),
            pd.Timestamp(y+1,4,15),pd.Timestamp(y+1,5,15),
        ]
        return min(abs((date - c).days) for c in candidates)
    df['peak_proximity'] = 1 / (1 + d.map(days_to_nearest_peak))
    month = d.dt.month
    df['month_sin'] = np.sin(2 * np.pi * month / 12)
    df['month_cos'] = np.cos(2 * np.pi * month / 12)
    for k in range(1, 5):
        df[f'doy_sin{k}'] = np.sin(2 * np.pi * k * df['doy'] / 365.25)
        df[f'doy_cos{k}'] = np.cos(2 * np.pi * k * df['doy'] / 365.25)
    dim = d.dt.days_in_month
    df['days_to_me'] = (dim - d.dt.day).astype(int)
    df['payday_pos'] = (dim - d.dt.day) / dim
    for n in [0, 1, 2, 3]:
        df[f'is_last{n}d'] = (df['days_to_me'] <= n).astype(int)
    df['is_first_day'] = (d.dt.day == 1).astype(int)
    df['is_mid_dip']   = ((d.dt.day >= 3) & (d.dt.day <= 6)).astype(int)
    df['is_weekend'] = (d.dt.dayofweek >= 5).astype(int)
    TET_DICT = {
        2010:'2010-02-14',2011:'2011-02-03',2012:'2012-01-23',
        2013:'2013-02-10',2014:'2014-01-31',2015:'2015-02-19',
        2016:'2016-02-08',2017:'2017-01-28',2018:'2018-02-16',
        2019:'2019-02-05',2020:'2020-01-25',2021:'2021-02-12',
        2022:'2022-02-01',2023:'2023-01-22',2024:'2024-02-10',
        2025:'2025-01-29',
    }
    TET = {y: pd.Timestamp(v) for y, v in TET_DICT.items()}
    def days_to_tet(date):
        y = date.year
        tt = TET.get(y); tn = TET.get(y + 1)
        if tt is not None and date <= tt: return (tt - date).days
        if tt is not None and tn is not None: return -(date - tt).days
        return 999
    td = df['days_to_tet'] = d.apply(days_to_tet)
    df['is_pre_tet6w'] = ((td >= 1) & (td <= 42)).astype(int)
    df['is_pre_tet2w'] = ((td >= 1) & (td <= 14)).astype(int)
    df['is_pre_tet1w'] = ((td >= 1) & (td <= 7)).astype(int)
    df['is_tet_week']  = ((td >= -3) & (td <= 3)).astype(int)
    df['is_post_tet']  = ((td >= -21) & (td <= -4)).astype(int)
    m, dy = df['month'], df['day']
    df['is_reunification'] = ((m == 4) & dy.between(28, 30)).astype(int)
    df['is_labor_day']     = ((m == 5) & dy.between(1, 3)).astype(int)
    df['is_national_day']  = ((m == 9) & dy.between(1, 4)).astype(int)
    df['is_valentine']     = ((m == 2) & (dy == 14)).astype(int)
    df['is_womens_38']     = ((m == 3) & (dy == 8)).astype(int)
    df['is_singles_1111']  = ((m == 11) & (dy == 11)).astype(int)
    df['is_1212']          = ((m == 12) & (dy == 12)).astype(int)
    df['is_ecom_day']      = ((m == dy) & m.between(6, 12)).astype(int)
    return df


In [84]:
def _get_sale_windows(years):
    windows = []
    for year in years:
        for s in SALE_SEASONS:
            try:
                start = pd.Timestamp(year=year, month=s['month'], day=s['start_day'])
            except ValueError:
                # start_day vượt quá số ngày trong tháng → dùng ngày cuối tháng
                start = pd.Timestamp(year=year, month=s['month'], day=1) + pd.offsets.MonthEnd(0)
            end = start + pd.Timedelta(days=s['duration'] - 1)
            windows.append((start, end, s['profit_rank']))
    return windows

In [85]:
def apply_monthly_bias_correction(pred_revenue, pred_cogs, test_dates, train_df):

    train_df = train_df.copy()
    train_df['Date'] = pd.to_datetime(train_df['Date'])
    
    # Dung 3 nam gan nhat de tinh correction factor
    last3_yrs = sorted(train_df['Date'].dt.year.unique())[-3:]
    tr3 = train_df[train_df['Date'].dt.year.isin(last3_yrs)]
    
    # Monthly mean Revenue tu training
    monthly_actual = tr3.groupby(tr3['Date'].dt.month)['Revenue'].mean()
    
    # Monthly mean tu prediction cua training period (proxy: dung Prophet trend * seasonal)
    # Thay the: dung ratio actual/lag365 de estimate error pattern
    tr3_prev = train_df[train_df['Date'].dt.year.isin([y-1 for y in last3_yrs])]
    if len(tr3_prev) > 0:
        date_rev = dict(zip(train_df['Date'], train_df['Revenue']))
        tr3 = tr3.copy()
        tr3['lag365'] = (tr3['Date'] - pd.Timedelta(days=365)).map(date_rev)
        tr3 = tr3.dropna(subset=['lag365'])
        monthly_ratio_actual = tr3.groupby(tr3['Date'].dt.month).apply(
            lambda x: x['Revenue'].mean() / x['lag365'].mean() if x['lag365'].mean() > 0 else 1.0
        )
        # Correction = scale predictions so monthly distribution matches training ratio
        pred_df = pd.DataFrame({'Revenue': pred_revenue, 'COGS': pred_cogs, 'Date': pd.to_datetime(test_dates)})
        pred_df['month'] = pred_df['Date'].dt.month
        
        # Soft correction: 50% blend (avoid overcorrection)
        for m in range(1, 13):
            mask = pred_df['month'] == m
            if mask.sum() == 0 or m not in monthly_ratio_actual.index:
                continue
            # Get lag365 for test period
            test_lag_dates = pred_df.loc[mask, 'Date'] - pd.Timedelta(days=365)
            test_lag_rev = test_lag_dates.map(date_rev)
            if test_lag_rev.isna().all():
                continue
            test_lag_mean = test_lag_rev.dropna().mean()
            if test_lag_mean <= 0:
                continue
            expected = test_lag_mean * monthly_ratio_actual[m]
            actual_pred = pred_df.loc[mask, 'Revenue'].mean()
            if actual_pred > 0:
                scale = expected / actual_pred
                # Soft: 40% correction
                scale = 1 + 0.4 * (scale - 1)
                scale = np.clip(scale, 0.7, 1.4)  # cap correction at ±30%
                pred_df.loc[mask, 'Revenue'] *= scale
                pred_df.loc[mask, 'COGS']   *= scale
    
    return pred_df['Revenue'].values, pred_df['COGS'].values

In [86]:
def add_sale_features(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])

    # ── Sale season windows ─────────────────────────────────────────────────
    years   = df['Date'].dt.year.unique()
    windows = _get_sale_windows(years)

    # Khởi tạo
    df['is_sale_season']      = 0
    df['sale_rank']           = 0      # 0 = không sale
    df['sale_progress']       = 0.0   # 0→1 trong đợt sale
    df['days_to_next_sale']   = 999
    df['days_since_last_sale']= 999

    dates = df['Date'].values  # numpy array để vectorise

    for start, end, rank in windows:
        mask_in = (df['Date'] >= start) & (df['Date'] <= end)

        # is_sale_season & sale_rank
        df.loc[mask_in, 'is_sale_season'] = 1
        df.loc[mask_in, 'sale_rank']      = rank

        # days_to_next_sale: cập nhật cho các ngày TRƯỚC đợt sale này
        mask_before = df['Date'] < start
        days_to = (start - df.loc[mask_before, 'Date']).dt.days
        df.loc[mask_before, 'days_to_next_sale'] = np.minimum(
            df.loc[mask_before, 'days_to_next_sale'], days_to
        )

        # days_since_last_sale: cập nhật cho các ngày SAU đợt sale này
        mask_after = df['Date'] > end
        days_since = (df.loc[mask_after, 'Date'] - end).dt.days
        df.loc[mask_after, 'days_since_last_sale'] = np.minimum(
            df.loc[mask_after, 'days_since_last_sale'], days_since
        )

    # Với ngày đang trong sale: days_to = 0, days_since = 0
    df.loc[df['is_sale_season'] == 1, 'days_to_next_sale']    = 0
    df.loc[df['is_sale_season'] == 1, 'days_since_last_sale'] = 0

    return df

In [87]:
def add_features(df):
    df = df.copy()
    df = add_seasonality_features(df)
    df = add_sale_features(df)

    return df

In [88]:
def tune_base_models(train_path):
    print("1. Chuẩn bị dữ liệu và Pre-calculate Folds...")
    df = pd.read_csv(train_path)
    df['Revenue_log'] = np.log1p(df['Revenue'])
    df['COGS_Ratio'] = df['COGS'] / (df['Revenue'] + 1e-9)
    df = add_features(df)
    
    static_features = [
        'year', 'month', 'day', 'dayofweek', 'doy', 'is_weekend',
        'is_peak_season', 'peak_proximity',
        'month_sin', 'month_cos',
        'doy_sin1', 'doy_cos1', 'doy_sin2', 'doy_cos2',
        'doy_sin3', 'doy_cos3', 'doy_sin4', 'doy_cos4',
        'days_to_me', 'payday_pos',
        'is_last0d', 'is_last1d', 'is_last2d', 'is_last3d',
        'is_first_day', 'is_mid_dip',
        'days_to_tet', 'is_pre_tet6w', 'is_pre_tet2w', 'is_pre_tet1w',
        'is_tet_week', 'is_post_tet',
        'is_reunification', 'is_labor_day', 'is_national_day',
        'is_valentine', 'is_womens_38',
        'is_singles_1111', 'is_1212', 'is_ecom_day',
        'is_sale_season', 'sale_rank', 'days_to_next_sale', 'days_since_last_sale',
    ]
    
    folds = [
        ('2012-07-04', '2018-12-31', '2019-01-01', '2019-12-31'),
        ('2012-07-04', '2019-12-31', '2020-01-01', '2020-12-31'),
        ('2012-07-04', '2020-12-31', '2021-01-01', '2021-12-31'),
        ('2012-07-04', '2021-12-31', '2022-01-01', '2022-12-31')
    ]
    
    cached_folds = []
    for train_start, train_end, val_start, val_end in folds:
        train_fold = df[(df['Date'] >= train_start) & (df['Date'] <= train_end)].copy()
        val_fold = df[(df['Date'] >= val_start) & (df['Date'] <= val_end)].copy()
        
        prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        prophet_df = train_fold[['Date', 'Revenue_log']].rename(columns={'Date': 'ds', 'Revenue_log': 'y'})
        prophet.fit(prophet_df)
        
        train_fold['residual'] = train_fold['Revenue_log'] - prophet.predict(prophet_df)['yhat'].values
        val_trend = prophet.predict(val_fold[['Date']].rename(columns={'Date': 'ds'}))['yhat'].values
        val_fold['residual'] = val_fold['Revenue_log'] - val_trend
        
        features = static_features 
        cached_folds.append({
            'X_train': train_fold[features], 'y_train_res': train_fold['residual'],
            'X_val': val_fold[features], 'y_val_res': val_fold['residual'],
            'val_trend': val_trend, 'y_val_true_rev': val_fold['Revenue']
        })

    # ==========================================
    # TUNE XGBOOST (HUBER LOSS)
    # ==========================================
    def objective_xgb(trial):
        params = {
            'n_estimators': 2000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 8),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'objective': 'reg:pseudohubererror', 'device': 'cuda', 'tree_method': 'hist', 'random_state': 42
        }
        cv_maes = []
        for fold in cached_folds:
            model = xgb.XGBRegressor(**params, early_stopping_rounds=50)
            model.fit(fold['X_train'], fold['y_train_res'], eval_set=[(fold['X_val'], fold['y_val_res'])], verbose=False)
            val_pred_rev = np.expm1(fold['val_trend'] + model.predict(fold['X_val'])) 
            cv_maes.append(mean_absolute_error(fold['y_val_true_rev'], val_pred_rev))
        return np.mean(cv_maes)

    # ==========================================
    # TUNE LIGHTGBM (HUBER LOSS)
    # ==========================================
    def objective_lgb(trial):
        params = {
            'n_estimators': 2000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 8),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'objective': 'huber', 'device': 'gpu', 'random_state': 42, 'verbose': -1
        }
        cv_maes = []
        for fold in cached_folds:
            model = lgb.LGBMRegressor(**params)
            model.fit(fold['X_train'], fold['y_train_res'], eval_set=[(fold['X_val'], fold['y_val_res'])], callbacks=[lgb.early_stopping(50, verbose=False)])
            val_pred_rev = np.expm1(fold['val_trend'] + model.predict(fold['X_val']))
            cv_maes.append(mean_absolute_error(fold['y_val_true_rev'], val_pred_rev))
        return np.mean(cv_maes)

    # ==========================================
    # TUNE CATBOOST (HUBER LOSS)
    # ==========================================
    def objective_cat(trial):
        params = {
            'iterations': 2000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 8),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
            'loss_function': 'Huber:delta=1.5', 'task_type': 'GPU', 'random_state': 42, 'verbose': False
        }
        cv_maes = []
        for fold in cached_folds:
            model = CatBoostRegressor(**params, early_stopping_rounds=50)
            model.fit(fold['X_train'], fold['y_train_res'], eval_set=[(fold['X_val'], fold['y_val_res'])], verbose=False)
            val_pred_rev = np.expm1(fold['val_trend'] + model.predict(fold['X_val']))
            cv_maes.append(mean_absolute_error(fold['y_val_true_rev'], val_pred_rev))
        return np.mean(cv_maes)
    print("Đang Tune XGBoost...")
    study_xgb = optuna.create_study(direction="minimize")
    study_xgb.optimize(objective_xgb, n_trials=30)
    
    print("\nĐang Tune LightGBM...")
    study_lgb = optuna.create_study(direction="minimize")
    study_lgb.optimize(objective_lgb, n_trials=30)
    
    print("\nĐang Tune CatBoost...")
    study_cat = optuna.create_study(direction="minimize")
    study_cat.optimize(objective_cat, n_trials=30)

    print("\nXong.")
    print("\n[KẾT QUẢ REVENUE PARAMS ĐỂ COPY VÀO STACKING]")
    print(f"xgb_params_rev = {study_xgb.best_trial.params}")
    print(f"lgb_params_rev = {study_lgb.best_trial.params}")
    print(f"cat_params_rev = {study_cat.best_trial.params}")
    return study_xgb.best_trial.params, study_lgb.best_trial.params, study_cat.best_trial.params

In [89]:
best_xgb_params = {
    "n_estimators": 2000,
    "learning_rate": 0.07687359172082428,
    "max_depth": 4,
    "objective": "reg:pseudohubererror",
    #"device": "cuda",
    "tree_method": "hist",
    "random_state": 42,
    "subsample": 0.9640163894074474,
    "colsample_bytree": 0.9988728404218791,
    "min_child_weight": 6
}

best_lgb_params = {
    "n_estimators": 2000,
    "learning_rate": 0.010074446630670307,
    "max_depth": 4,
    "objective": "huber",
    #"device": "gpu",
    "random_state": 42,
    "verbose": -1,
    "num_leaves": 28,
    "subsample": 0.932752195699773,
    "colsample_bytree": 0.8528851999663052
}

best_cat_params = {
    "iterations": 2000,
    "learning_rate": 0.09865972508027333,
    "depth": 4,
    "loss_function": "Huber:delta=1.5",
    #"task_type": "GPU",
    "random_state": 42,
    "verbose": False,
    "l2_leaf_reg": 5.35578476395823
}

In [ ]:
from sklearn.linear_model import HuberRegressor
from sklearn.model_selection import GridSearchCV

In [90]:
def run_stacking_pipeline(train_path, test_path, best_xgb_params, best_lgb_params, best_cat_params):
    print("1. Data Prep & Deterministic Features...")
    df = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    df['Revenue_log'] = np.log1p(df['Revenue'])
    df['COGS_Ratio'] = df['COGS'] / (df['Revenue'] + 1e-9) 
    
    df = add_features(df)
    df_test = add_features(df_test)

    print(f"Train: {len(df.columns)} features")
    print(f"Test: {len(df_test.columns)} features")
    print()
    
    static_features = [
        'year', 'month', 'day', 'dayofweek', 'doy', 'is_weekend',
        'is_peak_season', 'peak_proximity',
        'month_sin', 'month_cos',
        'doy_sin1', 'doy_cos1', 'doy_sin2', 'doy_cos2',
        'doy_sin3', 'doy_cos3', 'doy_sin4', 'doy_cos4',
        'days_to_me', 'payday_pos',
        'is_last0d', 'is_last1d', 'is_last2d', 'is_last3d',
        'is_first_day', 'is_mid_dip',
        'days_to_tet', 'is_pre_tet6w', 'is_pre_tet2w', 'is_pre_tet1w',
        'is_tet_week', 'is_post_tet',
        'is_reunification', 'is_labor_day', 'is_national_day',
        'is_valentine', 'is_womens_38',
        'is_singles_1111', 'is_1212', 'is_ecom_day',
        'is_sale_season', 'sale_rank', 'days_to_next_sale', 'days_since_last_sale',
    ]
    
    folds = [
        ('2012-07-04', '2018-12-31', '2019-01-01', '2019-12-31'),
        ('2012-07-04', '2019-12-31', '2020-01-01', '2020-12-31'),
        ('2012-07-04', '2020-12-31', '2021-01-01', '2021-12-31'),
        ('2012-07-04', '2021-12-31', '2022-01-01', '2022-12-31')
    ]
    
    # Cấu hình Base Models (Tất cả dùng Huber Loss)
    xgb_params = {'n_estimators': 2000, 'learning_rate': 0.03, 'max_depth': 6, 'objective': 'reg:pseudohubererror', 'tree_method': 'hist', 'random_state': 42}
    lgb_params = {'n_estimators': 2000, 'learning_rate': 0.03, 'max_depth': 6, 'objective': 'huber', 'random_state': 42, 'verbose': -1}
    cat_params = {'iterations': 2000, 'learning_rate': 0.03, 'depth': 6, 'loss_function': 'Huber:delta=1.5', 'random_state': 42, 'verbose': False}
    
    xgb_params.update(best_xgb_params)
    lgb_params.update(best_lgb_params)
    cat_params.update(best_cat_params)

    print("Base models's params:")
    print(f"best_xgb_params = {xgb_params}")
    print(f"best_lgb_params = {lgb_params}")
    print(f"best_cat_params = {cat_params}")
    print()
    
    # Arrays thu thập dự đoán Out-Of-Fold (OOF) để huấn luyện Trọng tài
    oof_rev_xgb, oof_rev_lgb, oof_rev_cat = [], [], []
    oof_rat_xgb, oof_rat_lgb, oof_rat_cat = [], [], []
    oof_y_rev, oof_y_rat = [], []
    
    print("2. Walk-Forward CV & Generating OOF Predictions...")
    for i, (train_start, train_end, val_start, val_end) in enumerate(folds):
        train_fold = df[(df['Date'] >= train_start) & (df['Date'] <= train_end)].copy()
        val_fold = df[(df['Date'] >= val_start) & (df['Date'] <= val_end)].copy()
        
        prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        prophet_df = train_fold[['Date', 'Revenue_log']].rename(columns={'Date': 'ds', 'Revenue_log': 'y'})
        prophet.fit(prophet_df)
        
        train_trend = prophet.predict(prophet_df)['yhat'].values
        val_trend = prophet.predict(val_fold[['Date']].rename(columns={'Date': 'ds'}))['yhat'].values
        train_fold['residual'] = train_fold['Revenue_log'] - train_trend
        val_fold['residual'] = val_fold['Revenue_log'] - val_trend
        
        features = static_features 
        X_tr, X_va = train_fold[features], val_fold[features]
        
        # --- TRAIN BASE MODELS (REVENUE) ---
        m_rev_xgb = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
        m_rev_xgb.fit(X_tr, train_fold['residual'], eval_set=[(X_va, val_fold['residual'])], verbose=False)
        
        m_rev_lgb = lgb.LGBMRegressor(**lgb_params)
        m_rev_lgb.fit(X_tr, train_fold['residual'], eval_set=[(X_va, val_fold['residual'])], callbacks=[lgb.early_stopping(50, verbose=False)])
        
        m_rev_cat = CatBoostRegressor(**cat_params, early_stopping_rounds=50)
        m_rev_cat.fit(X_tr, train_fold['residual'], eval_set=[(X_va, val_fold['residual'])], verbose=False)
        
        # Thu thập log OOF Revenue 
        log_p_xgb_rev = val_trend + m_rev_xgb.predict(X_va)
        log_p_lgb_rev = val_trend + m_rev_lgb.predict(X_va)
        log_p_cat_rev = val_trend + m_rev_cat.predict(X_va)
        
        oof_rev_xgb.extend(log_p_xgb_rev)
        oof_rev_lgb.extend(log_p_lgb_rev)
        oof_rev_cat.extend(log_p_cat_rev)
        oof_y_rev.extend(val_fold['Revenue_log'])
        
        # --- TRAIN BASE MODELS (COGS RATIO) ---
        m_rat_xgb = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
        m_rat_xgb.fit(X_tr, train_fold['COGS_Ratio'], eval_set=[(X_va, val_fold['COGS_Ratio'])], verbose=False)
        
        m_rat_lgb = lgb.LGBMRegressor(**lgb_params)
        m_rat_lgb.fit(X_tr, train_fold['COGS_Ratio'], eval_set=[(X_va, val_fold['COGS_Ratio'])], callbacks=[lgb.early_stopping(50, verbose=False)])
        
        m_rat_cat = CatBoostRegressor(**cat_params, early_stopping_rounds=50)
        m_rat_cat.fit(X_tr, train_fold['COGS_Ratio'], eval_set=[(X_va, val_fold['COGS_Ratio'])], verbose=False)
        
        # Thu thập OOF Ratio
        oof_rat_xgb.extend(m_rat_xgb.predict(X_va))
        oof_rat_lgb.extend(m_rat_lgb.predict(X_va))
        oof_rat_cat.extend(m_rat_cat.predict(X_va))
        oof_y_rat.extend(val_fold['COGS_Ratio'])

    
    print("3. Training Meta-Models (HuberRegressor)...")
    X_meta_rev = np.column_stack([oof_rev_xgb, oof_rev_lgb, oof_rev_cat])
    # Revenue Meta Tuning
    huber_grid = {'epsilon': [1.2, 1.25, 1.35, 1.4, 1.45, 1.5], 'alpha': [0.0001, 0.001, 0.01, 0.1]}
    meta_rev_search = GridSearchCV(HuberRegressor(fit_intercept=False), huber_grid, cv=5, scoring='neg_mean_absolute_error')
    meta_rev_search.fit(X_meta_rev, oof_y_rev)
    meta_rev_model = meta_rev_search.best_estimator_
    
    X_meta_rat = np.column_stack([oof_rat_xgb, oof_rat_lgb, oof_rat_cat])
    # Ratio Meta Tuning (Epsilon nhỏ hơn vì khoảng giá trị nhỏ)
    huber_ratio_grid = {'epsilon': [1.01, 1.05, 1.1, 1.2], 'alpha': [0.0001, 0.001, 0.01, 0.1]}
    meta_rat_search = GridSearchCV(HuberRegressor(fit_intercept=False), huber_ratio_grid, cv=5, scoring='neg_mean_absolute_error')
    meta_rat_search.fit(X_meta_rat, oof_y_rat)
    meta_rat_model = meta_rat_search.best_estimator_
    
    print(f" -> Trọng số (XGB/LGB/CAT) do Meta Model cấp - Revenue: {meta_rev_model.coef_}")
    print(f" -> Trọng số (XGB/LGB/CAT) do Meta Model cấp - Ratio: {meta_rat_model.coef_}")

    
    print("4. Training Full Base Models for Test Set...")
    
    final_features = static_features
    
    # Bỏ early_stopping
    xgb_params.pop('early_stopping_rounds', None)
    cat_params.pop('early_stopping_rounds', None)
    xgb_params['n_estimators'] = 600
    lgb_params['n_estimators'] = 600
    cat_params['iterations'] = 600
    
    final_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    prophet_df_full = df[['Date', 'Revenue_log']].rename(columns={'Date': 'ds', 'Revenue_log': 'y'})
    final_prophet.fit(prophet_df_full)
    df['residual'] = df['Revenue_log'] - final_prophet.predict(prophet_df_full)['yhat'].values
    
    # Train Revenue Full
    f_rev_xgb, f_rev_lgb, f_rev_cat = xgb.XGBRegressor(**xgb_params), lgb.LGBMRegressor(**lgb_params), CatBoostRegressor(**cat_params)
    f_rev_xgb.fit(df[final_features], df['residual'], verbose=False)
    f_rev_lgb.fit(df[final_features], df['residual'])
    f_rev_cat.fit(df[final_features], df['residual'], verbose=False)
    
    # Train Ratio Full
    f_rat_xgb, f_rat_lgb, f_rat_cat = xgb.XGBRegressor(**xgb_params), lgb.LGBMRegressor(**lgb_params), CatBoostRegressor(**cat_params)
    f_rat_xgb.fit(df[final_features], df['COGS_Ratio'], verbose=False)
    f_rat_lgb.fit(df[final_features], df['COGS_Ratio'])
    f_rat_cat.fit(df[final_features], df['COGS_Ratio'], verbose=False)
    
    print("5. Generating Meta-Predictions...")
    test_trend = final_prophet.predict(df_test[['Date']].rename(columns={'Date': 'ds'}))['yhat'].values
    
    # Base Predictions (Revenue)
    test_log_p_rev_xgb = test_trend + f_rev_xgb.predict(df_test[final_features])
    test_log_p_rev_lgb = test_trend + f_rev_lgb.predict(df_test[final_features])
    test_log_p_rev_cat = test_trend + f_rev_cat.predict(df_test[final_features])
    
    # Base Predictions (Ratio)
    test_p_rat_xgb = f_rat_xgb.predict(df_test[final_features])
    test_p_rat_lgb = f_rat_lgb.predict(df_test[final_features])
    test_p_rat_cat = f_rat_cat.predict(df_test[final_features])
    
    # Meta Model Inference
    X_test_meta_rev = np.column_stack([test_log_p_rev_xgb, test_log_p_rev_lgb, test_log_p_rev_cat])
    X_test_meta_rat = np.column_stack([test_p_rat_xgb, test_p_rat_lgb, test_p_rat_cat])
    
    final_log_rev = meta_rev_model.predict(X_test_meta_rev)
    df_test['Revenue'] = np.expm1(final_log_rev)
    df_test['COGS_Ratio_Pred'] = meta_rat_model.predict(X_test_meta_rat)
    
    df_test['COGS'] = df_test['Revenue'] * df_test['COGS_Ratio_Pred']
    df_test['Revenue'] = df_test['Revenue'].clip(lower=0)
    df_test['COGS'] = df_test['COGS'].clip(lower=0)
    
    submission_name = "submission.csv"
    submission = df_test[['Date', 'Revenue', 'COGS']]
    submission.to_csv(submission_name, index=False)
    print(f"Done! Saved to {submission_name}")

In [91]:
run_stacking_pipeline(
    DIR + "sales.csv", 
    DIR + "sample_submission.csv", 
    best_xgb_params, best_lgb_params, best_cat_params
)

1. Data Prep & Deterministic Features...


04:50:44 - cmdstanpy - INFO - Chain [1] start processing


Train: 50 features
Test: 48 features

Base models's params:
best_xgb_params = {'n_estimators': 2000, 'learning_rate': 0.07687359172082428, 'max_depth': 4, 'objective': 'reg:pseudohubererror', 'tree_method': 'hist', 'random_state': 42, 'subsample': 0.9640163894074474, 'colsample_bytree': 0.9988728404218791, 'min_child_weight': 6}
best_lgb_params = {'n_estimators': 2000, 'learning_rate': 0.010074446630670307, 'max_depth': 4, 'objective': 'huber', 'random_state': 42, 'verbose': -1, 'num_leaves': 28, 'subsample': 0.932752195699773, 'colsample_bytree': 0.8528851999663052}
best_cat_params = {'iterations': 2000, 'learning_rate': 0.09865972508027333, 'depth': 4, 'loss_function': 'Huber:delta=1.5', 'random_state': 42, 'verbose': False, 'l2_leaf_reg': 5.35578476395823}

2. Walk-Forward CV & Generating OOF Predictions...


04:50:44 - cmdstanpy - INFO - Chain [1] done processing
04:50:46 - cmdstanpy - INFO - Chain [1] start processing
04:50:47 - cmdstanpy - INFO - Chain [1] done processing
04:50:49 - cmdstanpy - INFO - Chain [1] start processing
04:50:49 - cmdstanpy - INFO - Chain [1] done processing
04:50:52 - cmdstanpy - INFO - Chain [1] start processing
04:50:52 - cmdstanpy - INFO - Chain [1] done processing


3. Training Meta-Models (HuberRegressor)...


04:50:58 - cmdstanpy - INFO - Chain [1] start processing


 -> Trọng số (XGB/LGB/CAT) do Meta Model cấp - Revenue: [0.30040571 0.63075365 0.07400793]
 -> Trọng số (XGB/LGB/CAT) do Meta Model cấp - Ratio: [-1.0412243  -0.07100739  2.24896105]
4. Training Full Base Models for Test Set...


04:51:00 - cmdstanpy - INFO - Chain [1] done processing


5. Generating Meta-Predictions...
Done! Saved to submission.csv
